# IMF Data Download via SDMX API

This notebook demonstrates how to access IMF data using the `sdmx1` Python library.

**Relevant for thesis:** Sovereign credit risk, oil-exporting emerging markets, macroeconomic indicators.

## 1. Setup and Installation

In [ ]:
# Install required packages (run once)
# !pip install sdmx1 pandas matplotlib

In [ ]:
import sdmx
import pandas as pd
import matplotlib.pyplot as plt

# Initialize IMF client
IMF = sdmx.Client('IMF_DATA')
print("IMF SDMX client initialized successfully")

## 2. Explore Available Datasets

The IMF provides many datasets. Let's see what's available.

In [ ]:
# Get list of available dataflows (datasets)
dataflows = IMF.dataflow()

# Convert to DataFrame for easier viewing
df_flows = pd.DataFrame([
    {'id': k, 'name': str(v.name)}
    for k, v in dataflows.dataflow.items()
])

print(f"Total available datasets: {len(df_flows)}")
df_flows.head(20)

In [ ]:
# Search for specific datasets by keyword
def search_datasets(keyword):
    """Search IMF datasets by keyword in name or ID."""
    mask = (
        df_flows['name'].str.lower().str.contains(keyword.lower()) |
        df_flows['id'].str.lower().str.contains(keyword.lower())
    )
    return df_flows[mask]

# Examples relevant to your thesis:
print("=== Datasets related to 'debt' ===")
display(search_datasets('debt'))

print("\n=== Datasets related to 'balance' ===")
display(search_datasets('balance'))

print("\n=== Datasets related to 'fiscal' ===")
display(search_datasets('fiscal'))

## 3. Explore Dataset Structure

Before querying, we need to understand the dataset's dimensions and codes.

In [ ]:
def explore_dataset(dataset_id):
    """Get the structure of a dataset (dimensions, codelists)."""
    try:
        # Get datastructure
        dsd = IMF.datastructure(dataset_id)
        
        # Get the structure definition
        structure = list(dsd.structure.values())[0]
        
        print(f"Dataset: {dataset_id}")
        print(f"\nDimensions:")
        for dim in structure.dimensions:
            print(f"  - {dim.id}: {dim.local_representation}")
        
        return dsd
    except Exception as e:
        print(f"Error exploring {dataset_id}: {e}")
        return None

# Explore a commonly used dataset
explore_dataset('IFS')  # International Financial Statistics

In [ ]:
def get_codelist(dataset_id, dimension_id):
    """Get available codes for a specific dimension."""
    try:
        dsd = IMF.datastructure(dataset_id)
        structure = list(dsd.structure.values())[0]
        
        # Find the dimension
        for dim in structure.dimensions:
            if dim.id == dimension_id:
                codelist = dim.local_representation.enumerated
                if codelist:
                    codes = pd.DataFrame([
                        {'code': k, 'name': str(v.name)}
                        for k, v in codelist.items()
                    ])
                    return codes
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None

# Example: Get country codes
# countries = get_codelist('IFS', 'REF_AREA')
# countries.head(20)

## 4. Download Data - Basic Examples

In [ ]:
def download_imf_data(dataset_id, key, start_period=None, end_period=None):
    """
    Download data from IMF.
    
    Parameters:
    -----------
    dataset_id : str
        The dataset identifier (e.g., 'IFS', 'BOP', 'GFS')
    key : str
        The series key (dimensions separated by '.')
        Use '+' for multiple values in a dimension
    start_period : str, optional
        Start period (e.g., '2000')
    end_period : str, optional
        End period (e.g., '2024')
    
    Returns:
    --------
    pandas.DataFrame
    """
    params = {}
    if start_period:
        params['startPeriod'] = start_period
    if end_period:
        params['endPeriod'] = end_period
    
    try:
        data_msg = IMF.data(dataset_id, key=key, params=params if params else None)
        df = sdmx.to_pandas(data_msg)
        
        # Reset index if it's a MultiIndex
        if isinstance(df, pd.Series):
            df = df.reset_index()
            df.columns = list(df.columns[:-1]) + ['value']
        
        return df
    except Exception as e:
        print(f"Error downloading data: {e}")
        return None

In [ ]:
# Example 1: CPI data for USA and Canada
cpi_data = download_imf_data(
    dataset_id='CPI',
    key='USA+CAN.CPI.CP01.IX.M',
    start_period='2018'
)

if cpi_data is not None:
    print("CPI Data:")
    display(cpi_data.head(10))

## 5. Data Relevant for Sovereign Credit Risk Thesis

Key datasets for your research:
- **IFS**: International Financial Statistics (reserves, exchange rates, etc.)
- **BOP**: Balance of Payments (current account, trade)
- **GFS**: Government Finance Statistics (fiscal data)
- **DOTS**: Direction of Trade Statistics (export data)

In [ ]:
# Define oil-exporting countries for your thesis
# ISO 3-letter codes
OIL_EXPORTERS = [
    'SAU',  # Saudi Arabia
    'RUS',  # Russia
    'ARE',  # UAE
    'KWT',  # Kuwait
    'NGA',  # Nigeria
    'AGO',  # Angola
    'IRQ',  # Iraq
    'KAZ',  # Kazakhstan
    'QAT',  # Qatar
    'COL',  # Colombia
    'ECU',  # Ecuador
    'VEN',  # Venezuela
    'OMN',  # Oman
    'BHR',  # Bahrain
]

# Control group - non-oil emerging markets
CONTROL_COUNTRIES = [
    'BRA',  # Brazil
    'MEX',  # Mexico
    'ZAF',  # South Africa
    'TUR',  # Turkey
    'IDN',  # Indonesia
    'PHL',  # Philippines
    'THA',  # Thailand
    'MYS',  # Malaysia
    'CHL',  # Chile
    'PER',  # Peru
]

ALL_COUNTRIES = OIL_EXPORTERS + CONTROL_COUNTRIES
print(f"Oil exporters: {len(OIL_EXPORTERS)}")
print(f"Control countries: {len(CONTROL_COUNTRIES)}")

In [ ]:
# Example: Download International Financial Statistics
# Foreign Exchange Reserves

# Note: The exact key structure depends on the dataset
# You may need to explore the structure first

# Try downloading reserves data for a few countries
try:
    # This is an example - actual key format varies by dataset
    reserves_data = IMF.data(
        'IFS',
        key='SAU+RUS+COL.RAFA_USD.A',  # Annual reserves in USD
        params={'startPeriod': '2000'}
    )
    reserves_df = sdmx.to_pandas(reserves_data)
    print("Reserves data downloaded successfully")
    display(reserves_df.head())
except Exception as e:
    print(f"Note: You may need to adjust the key format. Error: {e}")
    print("Use explore_dataset('IFS') to see available dimensions")

## 6. Batch Download Function

For downloading multiple indicators across countries.

In [ ]:
def batch_download(dataset_id, countries, indicator, frequency='A', 
                   start_period='2000', end_period='2024'):
    """
    Download data for multiple countries.
    
    Parameters:
    -----------
    dataset_id : str
        Dataset ID
    countries : list
        List of country codes
    indicator : str
        Indicator code
    frequency : str
        'A' for annual, 'Q' for quarterly, 'M' for monthly
    
    Returns:
    --------
    pandas.DataFrame with countries as columns
    """
    all_data = []
    
    for country in countries:
        try:
            # Construct key - adjust format as needed for specific datasets
            key = f"{country}.{indicator}.{frequency}"
            
            data_msg = IMF.data(
                dataset_id, 
                key=key,
                params={'startPeriod': start_period, 'endPeriod': end_period}
            )
            
            df = sdmx.to_pandas(data_msg)
            if isinstance(df, pd.Series):
                df = df.reset_index()
                df['country'] = country
                all_data.append(df)
                print(f"✓ {country}: {len(df)} observations")
            
        except Exception as e:
            print(f"✗ {country}: {str(e)[:50]}")
    
    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return None

## 7. Save Downloaded Data

In [ ]:
def save_data(df, filename, output_dir='./data/imf'):
    """Save DataFrame to CSV and Excel."""
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    # Save as CSV
    csv_path = os.path.join(output_dir, f"{filename}.csv")
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")
    
    # Save as Excel
    xlsx_path = os.path.join(output_dir, f"{filename}.xlsx")
    df.to_excel(xlsx_path, index=False)
    print(f"Saved: {xlsx_path}")

# Example usage:
# if cpi_data is not None:
#     save_data(cpi_data, 'cpi_data')

## 8. Alternative: Direct API Access

If sdmx1 doesn't work for certain datasets, use direct HTTP requests.

In [ ]:
import requests

def download_via_rest(dataset_id, key, start_period=None, end_period=None):
    """
    Download IMF data via REST API directly.
    
    Alternative method if sdmx1 library has issues.
    """
    base_url = "https://sdmxcentral.imf.org/ws/public/sdmxapi/rest/data"
    
    url = f"{base_url}/{dataset_id}/{key}"
    
    params = {'format': 'csv'}  # Request CSV format for easier parsing
    if start_period:
        params['startPeriod'] = start_period
    if end_period:
        params['endPeriod'] = end_period
    
    headers = {
        'Accept': 'text/csv'
    }
    
    response = requests.get(url, params=params, headers=headers)
    
    if response.status_code == 200:
        from io import StringIO
        df = pd.read_csv(StringIO(response.text))
        return df
    else:
        print(f"Error {response.status_code}: {response.text[:200]}")
        return None

# Example:
# df = download_via_rest('CPI', 'USA.CPI.CP01.IX.M', start_period='2020')
# df.head()

## 9. Quick Reference: Useful IMF Datasets

| Dataset ID | Name | Relevant For |
|------------|------|-------------|
| IFS | International Financial Statistics | FX reserves, exchange rates |
| BOP | Balance of Payments | Current account, trade |
| GFS | Government Finance Statistics | Fiscal balance, debt |
| GFSR | Global Financial Stability Report | Financial indicators |
| DOT | Direction of Trade | Export/import data |
| WEO | World Economic Outlook | GDP, forecasts |
| CPI | Consumer Price Index | Inflation |
| PCPS | Primary Commodity Prices | Oil prices (alternative) |

## 10. Notes for Your Thesis

**Data you likely need:**
1. **Foreign exchange reserves** (IFS) - for CCA model
2. **External debt** (GFS or IFS) - sovereign liabilities
3. **GDP** (WEO) - for normalization
4. **Oil exports as % of total exports** (DOT or WITS) - oil dependence metric
5. **Fiscal balance** (GFS) - net fiscal assets

**Remember:** CDS spread data isn't from IMF - you'll get that from Bloomberg/Refinitiv or your existing EMBI dataset.

In [ ]:
# Template for your specific data needs
# Uncomment and modify as needed

# # 1. Download FX Reserves
# fx_reserves = download_imf_data('IFS', key='...', start_period='2000')
# save_data(fx_reserves, 'fx_reserves')

# # 2. Download External Debt
# external_debt = download_imf_data('GFS', key='...', start_period='2000')
# save_data(external_debt, 'external_debt')

# # 3. Combine into master dataset
# master_df = fx_reserves.merge(external_debt, on=['country', 'period'])
# save_data(master_df, 'imf_master_data')